In [1]:
!pip install torch torchvision torchaudio transformer_lens circuitsvis numpy==1.26.4 pandas==2.2.2


[notice] A new release of pip available: 22.2 -> 26.1.1
[notice] To update, run: C:\Users\anant\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch
import transformer_lens

# Check if your computer can use a GPU (not required for this, but good to know)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Running on: {device}")

# If this prints without error, your 'lab' is officially open.

C:\Users\anant\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.12.0+cpu
Running on: cpu


In [3]:
# Save this as transformer_lab.ipynb or research.py
import torch
import numpy as np
from transformer_lens import HookedTransformer
import circuitsvis as cv
import webbrowser
import os

# 1. Load the Model
# 'gpt2-small' is a 12-layer transformer. Each layer has 12 attention heads.
print("Loading model... this may take a minute.")
model = HookedTransformer.from_pretrained("gpt2-small")

# 2. Define the prompt
# We want to see how the model connects "College" to "Station"
prompt = "The university is located in College Station, Texas."

# 3. Run the model and cache the internal math (the 'activations')
# This is like taking an MRI of the AI while it thinks.
logits, cache = model.run_with_cache(prompt)

# 4. Get the tokens (the words as the AI sees them)
tokens = model.to_str_tokens(prompt)

# 5. Extract Attention Patterns
# Let's look at Layer 5—a layer often responsible for factual associations.
layer_to_inspect = 5
attention_pattern = cache["pattern", layer_to_inspect]

# 6. Generate the Visualization
# This creates an interactive HTML/JavaScript object
print(f"Generating visualization for Layer {layer_to_inspect}...")
viz = cv.attention.attention_heads(tokens=tokens, attention=attention_pattern)

# --- HOW TO VIEW ---
# If using a Notebook (.ipynb), simply typing 'viz' displays it:
# viz 

# If using a Python script (.py), we save it to an HTML file and open it:
html_file = "attention_map.html"
with open(html_file, "w") as f:
    f.write(str(viz))

print(f"Visualization saved to {os.path.abspath(html_file)}")
webbrowser.open('file://' + os.path.abspath(html_file))

# 7. PROBING THE MATH (Optional Console Output)
# Let's look at the raw Query vector for the word 'Station' in Head 0
# Shape: [batch, pos, head, d_head]
q_vector = cache["q", layer_to_inspect][0, -2, 0, :] 
print(f"\n[Math Check] Query vector for 'Station' (first 5 dims): {q_vector[:5].tolist()}")

Loading model... this may take a minute.


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3929.31it/s]


Loaded pretrained model gpt2-small into HookedTransformer
Generating visualization for Layer 5...
Visualization saved to c:\TransfomerPractice\attention_map.html

[Math Check] Query vector for 'Station' (first 5 dims): [1.8582020998001099, 1.4201477766036987, 3.560563564300537, 0.479188472032547, -1.2693159580230713]


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create the UI elements
text_input = widgets.Text(value=prompt, description='Prompt:', style={'description_width': 'initial'})
layer_slider = widgets.IntSlider(value=5, min=0, max=11, description='Layer:')
button = widgets.Button(description="Update Map", button_style='success')
output_area = widgets.Output()

def update_viz(b):
    with output_area:
        clear_output(wait=True)
        new_prompt = text_input.value
        new_layer = layer_slider.value
        
        # Re-run the model math
        logits, cache = model.run_with_cache(new_prompt)
        tokens = model.to_str_tokens(new_prompt)
        attention_pattern = cache["pattern", new_layer]
        
        # Display the visualizer
        display(cv.attention.attention_heads(tokens=tokens, attention=attention_pattern))

button.on_click(update_viz)

# Display the dashboard
display(widgets.VBox([text_input, layer_slider, button]), output_area)

ModuleNotFoundError: No module named 'ipywidgets'